# Exact role-constrained Bayesian network — caou

This notebook reports the exact A0 → A1 → B → C Bayesian network and the
empirical recurrent-scenario analysis. Scenario selection remains independent
of the Bayesian network. Edge directions follow the predefined role ordering
and are not causal directions inferred from the data.


In [1]:
from pathlib import Path
import sys
import warnings
import pandas as pd
from IPython.display import Image, display

warnings.filterwarnings(
    "ignore",
    message=r"(?s).*support for the `google[.]generativeai` package has ended.*",
    category=FutureWarning,
)
SCENARIO_DIR = next(
    candidate
    for base in (Path.cwd(), *Path.cwd().parents)
    for candidate in (base / "text" / "recurrent_scenarios", base)
    if (candidate / "scenario_pipeline.py").is_file()
)
if str(SCENARIO_DIR) not in sys.path:
    sys.path.insert(0, str(SCENARIO_DIR))
from scenario_analysis import _output_paths, run_global_bn_scenario_mining
from scenario_pipeline import load_bn_analysis_config, load_selected_configurations


In [2]:
DATASET_ID = 'caou'
RUN_DIR = SCENARIO_DIR / "runs" / "theme_discovery_audit" / DATASET_ID
CONFIG_PATH = SCENARIO_DIR / "config.yaml"
RUN_ANALYSIS = False  # Use the Slurm job for the 500-replicate bootstrap.
config = load_bn_analysis_config(CONFIG_PATH, DATASET_ID, RUN_DIR)
selections = load_selected_configurations(RUN_DIR)
paths = _output_paths(RUN_DIR)
print("Dataset:", DATASET_ID)
print("Output:", paths["root"])
print("Structure optimization: exact within the role-constrained graph class")
print("Bootstrap resamples:", config["bayesian_networks"]["bn_structure_bootstrap"]["n_resamples"])
print("Selected partitions:", selections)


Dataset: caou
Output: C:\Users\aho\Documents\analysis factor project\SAFER\text\recurrent_scenarios\runs\theme_discovery_audit\caou\bn_results_exact
Structure optimization: exact within the role-constrained graph class
Bootstrap resamples: 500
Selected partitions: {'A0': 'A0_cfg_029', 'A1': 'A1_cfg_007', 'B': 'B_cfg_051', 'C': 'C_cfg_047'}


## 1. Accident-level matrix and exact structure


In [3]:
if RUN_ANALYSIS:
    analysis = run_global_bn_scenario_mining(config, RUN_DIR, selections)
if not (paths["network"] / "global_bn_summary.csv").is_file():
    raise FileNotFoundError("Exact BN outputs are absent. Launch jobs/run_recurrent_scenarios_bn_exact.sh first.")
summary = pd.read_csv(paths["network"] / "global_bn_summary.csv")
local_scores = pd.read_csv(paths["network"] / "bn_local_scores.csv")
selected_parent_sets = pd.read_csv(paths["network"] / "bn_selected_parent_sets.csv")
display(summary)
display(selected_parent_sets)
print("Local parent sets evaluated:", len(local_scores))


FileNotFoundError: Exact BN outputs are absent. Launch jobs/run_recurrent_scenarios_bn_exact.sh first.

## 2. Assess support of selected conditional-probability tables


In [ ]:
display(local_scores.sort_values(["child_factor", "rank"]).head(30))
support_diagnostics = pd.read_csv(paths["network"] / "bn_cpt_support_diagnostics.csv")
diagnostic = support_diagnostics.iloc[0]
print("Selected BN:")
print("Number of CPT rows:", diagnostic["n_cpt_rows"])
print("Rows with N = 0:", diagnostic["n_rows_N_eq_0"])
print("Rows with N <= 2:", diagnostic["n_rows_N_le_2"])
print("Rows with N <= 5:", diagnostic["n_rows_N_le_5"])
print("MLE estimates equal to 0:", diagnostic["n_MLE_equal_0"])
print("MLE estimates equal to 1:", diagnostic["n_MLE_equal_1"])
print("Minimum observed cell count:", diagnostic["minimum_observed_cell_count"])
print("Median observed cell count:", diagnostic["median_observed_cell_count"])
print("Final CPT estimator:", diagnostic["final_CPT_estimation"])
display(support_diagnostics)
final_cpts = pd.read_csv(paths["network"] / "bn_final_cpts.csv")
display(final_cpts.head(30))
if diagnostic["final_CPT_estimation"] == "Jeffreys":
    print("Jeffreys regularization was used because the configured support criterion was not met.")
    display(pd.read_csv(paths["network"] / "bn_mle_cpts.csv").head(30))


## 3. Structural bootstrap and conditional contrasts


In [ ]:
bootstrap = pd.read_csv(paths["network"] / "bn_bootstrap_edges.csv")
edges = pd.read_csv(paths["network"] / "bn_edges_full.csv")
contrasts = pd.read_csv(paths["network"] / "bn_conditional_contrasts.csv")
display(bootstrap.sort_values("selection_frequency", ascending=False).head(20))
display(edges.sort_values("bootstrap_frequency", ascending=False).head(20))
display(contrasts.head(30))
for name in ("global_bn_stable_dependencies.png", "bn_stability_vs_conditional_contrast.png"):
    path = paths["figures"] / name
    if path.is_file():
        display(Image(filename=str(path)))


## 4. Empirical recurrent scenarios


In [ ]:
candidates = pd.read_csv(paths["scenarios"] / "scenario_candidates_all.csv")
recurrent = pd.read_csv(paths["scenarios"] / "recurrent_scenarios_all.csv")
thresholds = pd.read_csv(paths["scenarios"] / "scenario_threshold_summary.csv")
print("Admissible configurations:", len(candidates))
print("Closed recurrent scenarios:", len(recurrent))
display(thresholds)
display(recurrent[["scenario_id", "upstream_labels", "B_label", "C_label", "scenario_accident_count", "scenario_support", "confidence", "lift"]].head(20))


## 5. Recurrence relative to the Bayesian-network background


In [ ]:
discrepancy = pd.read_csv(paths["scenarios"] / "scenario_bn_discrepancy.csv")
sensitivity = pd.read_csv(paths["scenarios"] / "scenario_bn_cpt_sensitivity.csv")
article = pd.read_csv(paths["scenarios"] / "scenarios_article_table.csv")
display(discrepancy.sort_values("BN_support_interestingness", ascending=False).head(20))
display(sensitivity[["scenario_id", "BN_CPT_estimation", "BN_MLE_support", "BN_Jeffreys_support", "BN_MLE_minus_Jeffreys_pp"]].head(20))
display(article)
path = paths["figures"] / "observed_vs_bn_implied_recurrence.png"
if path.is_file():
    display(Image(filename=str(path)))


## 6. Files for the manuscript


In [ ]:
for folder in (paths["network"], paths["scenarios"], paths["figures"]):
    print(folder)
    for path in sorted(folder.glob("*")):
        if path.is_file():
            print("  ", path.name)
